# 🏠 PropIQ — USA House Price Prediction
*A Complete, Production-Ready Machine Learning Pipeline*

---

**Author:** PropIQ ML Team  
**Dataset:** USA Real Estate Dataset (`realtor-data.csv`) + ZIP Enrichment (`uszips.xlsx`)  
**Target:** `price` (continuous regression)  
**Version:** 2.0 — Professional Edition

---

## 📋 Project Description

PropIQ is an end-to-end machine learning pipeline for predicting residential property prices across the United States. It ingests raw MLS-style listing data, enriches it with ZIP-code-level demographic and geographic data, engineers a rich feature set, selects the top 35 most predictive features, and trains gradient-boosting models with geographic cross-validation.

## 🎯 Objectives

| # | Objective |
|---|----------|
| 1 | Clean and preprocess raw real estate listings |
| 2 | Enrich data with ZIP-code demographics and geography |
| 3 | Engineer domain-specific features (luxury score, room ratios, etc.) |
| 4 | Select the top 35 features using XGBoost importance + optional PCA |
| 5 | Train XGBoost and LightGBM with GroupKFold CV by ZIP code |
| 6 | Tune hyperparameters and compare models |
| 7 | Save all artifacts for deployment in a Streamlit app |

---

## 🗺️ Pipeline Overview

| Stage | Section | Description |
|-------|---------|-------------|
| 1 | Install & Imports | Dependencies and configuration |
| 2 | Load Data | Main CSV + ZIP enrichment merge |
| 3 | Initial Audit | Shape, dtypes, missing values |
| 4 | EDA | Distributions, correlations, visualizations |
| 5 | Cleaning | Invalid rows, duplicates, type fixes |
| 6 | Missing Values | Indicators + hierarchical imputation |
| 7 | Feature Engineering | 15+ new features |
| 8 | Feature Selection | XGBoost importance → Top 35 + PCA comparison |
| 9 | Encoding & Scaling | Target encoding, OHE, RobustScaler |
| 10 | Model Training | XGBoost, LightGBM with GroupKFold |
| 11 | Hyperparameter Tuning | Optuna (50 trials) |
| 12 | Model Comparison | Metrics table + plots |
| 13 | Save Artifacts | All models, preprocessors, metadata |
| 14 | Prediction Function | Ready-to-use inference function |
| 15 | Streamlit Prep | App scaffold for deployment |

## 0️⃣ Install Dependencies

In [ ]:
# Install all required packages
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm \
             catboost optuna shap joblib openpyxl missingno scipy --quiet

## 1️⃣ Imports & Configuration

> All libraries are imported here. We use a fixed `RANDOM_STATE=42` for full reproducibility.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# ── Core ─────────────────────────────────────────────────────────────────────
import os
import json
import numpy as np
import pandas as pd
from scipy import stats

# ── Visualization ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno

# ── Sklearn ──────────────────────────────────────────────────────────────────
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.model_selection import (GroupKFold, KFold,
                                     train_test_split, cross_val_score)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.decomposition import PCA

# ── Boosting ─────────────────────────────────────────────────────────────────
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# ── Tuning & Explainability ───────────────────────────────────────────────────
import optuna
import shap
import joblib

optuna.logging.set_verbosity(optuna.logging.WARNING)
shap.initjs()

# ── Plot Style ───────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 5)})
PALETTE = ['#4C72B0', '#C44E52', '#55A868', '#8172B2', '#CCB974', '#64B5CD']

# ── Config ───────────────────────────────────────────────────────────────────
RANDOM_STATE   = 42
TOP_N_FEATURES = 35          # Number of features for the final model
N_SPLITS       = 5           # GroupKFold splits
np.random.seed(RANDOM_STATE)

# ── Output directory ─────────────────────────────────────────────────────────
os.makedirs('artifacts', exist_ok=True)

print('✅ All imports successful')
print(f'   XGBoost  : {__import__("xgboost").__version__}')
print(f'   LightGBM : {__import__("lightgbm").__version__}')
print(f'   Sklearn  : {__import__("sklearn").__version__}')

## 2️⃣ Load Dataset

### 2A. Main Real Estate Dataset

> Expected columns: `price`, `bed`, `bath`, `acre_lot`, `house_size`, `zip_code`, `city`, `state`, `prev_sold_date`, `street`
>
> **Dataset source:** [Kaggle — USA Real Estate Dataset](https://www.kaggle.com/datasets/ahmedshahriarsakib/usa-real-estate-dataset)

In [ ]:
# ── Load main dataset ─────────────────────────────────────────────────────────
MAIN_DATA_PATH = 'realtor-data.csv'

df = pd.read_csv(MAIN_DATA_PATH, low_memory=False)
print(f'✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Columns: {df.columns.tolist()}')

In [ ]:
# ── Fix zip_code format: ensure 5-digit zero-padded string ───────────────────
df['zip_code'] = (
    df['zip_code']
    .astype(str)
    .str.strip()
    .str.replace(r'\.0$', '', regex=True)   # remove .0 from float repr
    .str.zfill(5)                            # left-pad to 5 digits
)

print(f'ZIP sample : {df["zip_code"].head(8).tolist()}')
print(f'Unique ZIPs: {df["zip_code"].nunique():,}')

### 2B. Load ZIP Code Enrichment (`uszips.xlsx`)

> **Source:** SimpleMaps US ZIP codes — 33,782 entries with lat/lng, population, density, county, timezone.  
> Download: https://simplemaps.com/data/us-zips

In [ ]:
# ── Load ZIP enrichment table ─────────────────────────────────────────────────
zip_df = pd.read_excel('uszips.xlsx', dtype={'zip': str})

# Standardize ZIP
zip_df['zip_code'] = (
    zip_df['zip']
    .astype(str)
    .str.strip()
    .str.zfill(5)
)

# Select useful columns
ZIP_COLS = ['zip_code', 'lat', 'lng', 'population', 'density',
            'county_name', 'timezone', 'state_name']
zip_df = zip_df[ZIP_COLS].copy()
zip_df.rename(columns={
    'population': 'zip_population',
    'density':    'zip_density',
    'state_name': 'state_full'
}, inplace=True)
zip_df = zip_df.drop_duplicates(subset='zip_code')

print(f'ZIP table shape: {zip_df.shape}')
print(zip_df.head(3).to_string())

### 2C. Merge Main Dataset with ZIP Enrichment

In [ ]:
# ── Left-join on zip_code ─────────────────────────────────────────────────────
rows_before = len(df)
df = df.merge(zip_df, on='zip_code', how='left')

print(f'Rows before merge : {rows_before:,}')
print(f'Rows after  merge : {len(df):,}')
print(f'ZIP match rate    : {df["lat"].notna().mean():.2%}')

# Debug unmatched ZIPs
unmatched = df[df['lat'].isna()]['zip_code'].value_counts().head(10)
if len(unmatched):
    print(f'\nTop unmatched ZIPs:\n{unmatched}')

## 3️⃣ Initial Data Audit

> Before touching the data, we understand what we have: shapes, types, missing values, and duplicates.

In [ ]:
# ── Shape & Dtypes ────────────────────────────────────────────────────────────
print('=' * 60)
print(f'  SHAPE : {df.shape[0]:,} rows × {df.shape[1]} columns')
print('=' * 60)
print('\nDtype breakdown:')
print(df.dtypes.value_counts())
print('\nColumn dtypes:')
print(df.dtypes)

In [ ]:
# ── Missing Value Report ──────────────────────────────────────────────────────
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct':   (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('missing_pct', ascending=False)
missing = missing[missing['missing_count'] > 0]

print('MISSING VALUE REPORT')
print('=' * 40)
print(missing.to_string())

In [ ]:
# ── Duplicate Detection ───────────────────────────────────────────────────────
dup_cols = [c for c in ['street', 'zip_code', 'price', 'house_size'] if c in df.columns]
print(f'Exact duplicates (all cols)        : {df.duplicated().sum():,}')
print(f'Near-duplicates on {dup_cols}: {df.duplicated(subset=dup_cols).sum():,}')

In [ ]:
# ── Summary Statistics ────────────────────────────────────────────────────────
df.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T.style.background_gradient(cmap='Blues', subset=['mean','50%'])

## 4️⃣ Exploratory Data Analysis (EDA)

> Deep dive into distributions, correlations, and geographic patterns.

In [ ]:
# ── Price Distribution (raw & log scale) ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

price_clean = df['price'].dropna()
price_clean = price_clean[(price_clean > 10_000) & (price_clean < 10_000_000)]

axes[0].hist(price_clean, bins=80, color=PALETTE[0], edgecolor='white', alpha=0.85)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[0].set_title('Price Distribution (Raw)', fontweight='bold')
axes[0].set_xlabel('Price')

axes[1].hist(np.log1p(price_clean), bins=80, color=PALETTE[1], edgecolor='white', alpha=0.85)
axes[1].set_title('Log(Price+1) Distribution', fontweight='bold')
axes[1].set_xlabel('log(Price)')

top_states = df['state'].value_counts().head(10).index
state_data = df[df['state'].isin(top_states)][['state', 'price']].dropna()
state_data = state_data[state_data['price'].between(50_000, 3_000_000)]
state_data['state'] = state_data['state'].str.upper()

state_data.boxplot(column='price', by='state', ax=axes[2],
                   boxprops=dict(color=PALETTE[0]),
                   medianprops=dict(color=PALETTE[1], linewidth=2))
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[2].set_title('Price by State (Top 10)', fontweight='bold')
axes[2].set_xlabel('State')

plt.suptitle('🏷️ Price Distribution Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Price stats:\n{price_clean.describe().apply(lambda x: f"${x:,.0f}")}')

In [ ]:
# ── Numeric Feature Distributions ────────────────────────────────────────────
num_feats = ['bed', 'bath', 'house_size', 'acre_lot']
num_feats = [c for c in num_feats if c in df.columns]

fig, axes = plt.subplots(2, len(num_feats), figsize=(16, 8))

for i, col in enumerate(num_feats):
    col_data = df[col].dropna()
    q99 = col_data.quantile(0.99)
    col_clipped = col_data[col_data <= q99]

    axes[0, i].hist(col_clipped, bins=60, color=PALETTE[i % len(PALETTE)],
                    edgecolor='white', alpha=0.85)
    axes[0, i].set_title(f'{col} Distribution', fontweight='bold')

    axes[1, i].boxplot(col_clipped, vert=True,
                       boxprops=dict(color=PALETTE[i % len(PALETTE)]),
                       medianprops=dict(color='red', linewidth=2))
    axes[1, i].set_title(f'{col} Boxplot (p99)', fontweight='bold')

plt.suptitle('📊 Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────────────────────
corr_cols = [c for c in ['price', 'bed', 'bath', 'house_size', 'acre_lot',
                          'lat', 'lng', 'zip_population', 'zip_density'] if c in df.columns]
corr_df   = df[corr_cols].dropna()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_df.corr(), dtype=bool))
sns.heatmap(corr_df.corr(), mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('🔗 Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Price vs Key Features ─────────────────────────────────────────────────────
scatter_feats = [c for c in ['house_size', 'bed', 'bath', 'zip_density'] if c in df.columns]

fig, axes = plt.subplots(1, len(scatter_feats), figsize=(18, 4))

sample = df[df['price'].between(50_000, 3_000_000)].sample(min(8000, len(df)), random_state=RANDOM_STATE)

for i, col in enumerate(scatter_feats):
    col_data = sample[[col, 'price']].dropna()
    q99 = col_data[col].quantile(0.99)
    col_data = col_data[col_data[col] <= q99]

    axes[i].scatter(col_data[col], col_data['price'],
                    alpha=0.15, s=5, color=PALETTE[i % len(PALETTE)])
    z = np.polyfit(col_data[col], col_data['price'], 1)
    p = np.poly1d(z)
    xs = np.linspace(col_data[col].min(), col_data[col].max(), 100)
    axes[i].plot(xs, p(xs), color='red', linewidth=1.5, linestyle='--')

    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Price' if i == 0 else '')
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
    axes[i].set_title(f'Price vs {col}', fontweight='bold')

plt.suptitle('📈 Price vs Feature Scatter Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Geographic Price Heat Map (lat/lng bins) ──────────────────────────────────
if 'lat' in df.columns and 'lng' in df.columns:
    geo_sample = df[['lat', 'lng', 'price']].dropna()
    geo_sample = geo_sample[
        geo_sample['lat'].between(24, 50) &
        geo_sample['lng'].between(-125, -65) &
        geo_sample['price'].between(50_000, 3_000_000)
    ]

    fig, ax = plt.subplots(figsize=(14, 7))
    sc = ax.scatter(geo_sample['lng'], geo_sample['lat'],
                    c=np.log1p(geo_sample['price']),
                    cmap='YlOrRd', alpha=0.3, s=3)
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('log(Price)', rotation=270, labelpad=15)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('🗺️ Geographic Price Distribution (Continental USA)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Top 15 States by Median Price ────────────────────────────────────────────
if 'state' in df.columns:
    state_median = (
        df[df['price'].between(50_000, 5_000_000)]
        .groupby('state')['price']
        .median()
        .sort_values(ascending=False)
        .head(15)
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    state_median.plot(kind='bar', ax=ax, color=PALETTE[0], edgecolor='white')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
    ax.set_title('🏆 Top 15 States by Median House Price', fontweight='bold', fontsize=13)
    ax.set_xlabel('State')
    ax.set_ylabel('Median Price')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

## 5️⃣ Data Preprocessing & Cleaning

> Remove invalid records, fix types, and standardize formats. We log every step.

In [ ]:
rows_start = len(df)
print(f'Starting shape: {df.shape}')

In [ ]:
# ── Remove rows with invalid numeric values ───────────────────────────────────
if 'price'      in df.columns: df = df[df['price']      > 0]
if 'house_size' in df.columns: df = df[df['house_size'] > 0]
if 'bath'       in df.columns: df = df[df['bath']       >= 0]
if 'bed'        in df.columns: df = df[df['bed']        >= 0]
if 'acre_lot'   in df.columns: df = df[df['acre_lot']   >= 0]

print(f'Rows removed (invalid values): {rows_start - len(df):,}')

In [ ]:
# ── Remove duplicate listings ─────────────────────────────────────────────────
dup_cols = [c for c in ['street', 'zip_code', 'price', 'house_size'] if c in df.columns]
rows_before_dup = len(df)
df = df.drop_duplicates(subset=dup_cols, keep='first').reset_index(drop=True)
print(f'Rows removed (duplicates)    : {rows_before_dup - len(df):,}')

In [ ]:
# ── Outlier Treatment: Cap price at 1st-99th percentile ──────────────────────
p01 = df['price'].quantile(0.01)
p99 = df['price'].quantile(0.99)
rows_before_clip = len(df)
df = df[(df['price'] >= p01) & (df['price'] <= p99)]
print(f'Rows removed (price outliers p1-p99): {rows_before_clip - len(df):,}')
print(f'Price range after clip: ${p01:,.0f} – ${p99:,.0f}')

In [ ]:
# ── Cap house_size and acre_lot to 99th percentile ────────────────────────────
for col in ['house_size', 'acre_lot']:
    if col in df.columns:
        cap = df[col].quantile(0.99)
        df[col] = df[col].clip(upper=cap)
        print(f'{col} capped at {cap:,.1f}')

In [ ]:
# ── Standardize string columns ────────────────────────────────────────────────
for col in ['state', 'city', 'street']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()

df['zip_code'] = df['zip_code'].astype(str).str.zfill(5)

print(f'\n✅ Final cleaned shape: {df.shape}')
df.head(3)

## 6️⃣ Missing Value Analysis & Imputation

> We create binary "was_missing" indicators before imputation so the model can learn from missingness patterns.

In [ ]:
# ── Missingness Visualization ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sample_vis = df.sample(min(5000, len(df)), random_state=RANDOM_STATE)
msno.matrix(sample_vis, ax=axes[0], sparkline=False)
axes[0].set_title('Missingness Matrix (sample 5k rows)', fontsize=13, fontweight='bold')

miss_pct = df.isnull().mean().sort_values(ascending=False)
miss_pct = miss_pct[miss_pct > 0]
if len(miss_pct):
    axes[1].barh(miss_pct.index, miss_pct.values * 100, color=PALETTE[1])
    axes[1].set_xlabel('Missing %')
    axes[1].set_title('Missing % per Column', fontsize=13, fontweight='bold')
    for i, v in enumerate(miss_pct.values):
        axes[1].text(v * 100 + 0.3, i, f'{v:.1%}', va='center', fontsize=9)
else:
    axes[1].text(0.5, 0.5, 'No missing values!', ha='center', va='center',
                 fontsize=14, color='green', transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

In [ ]:
# ── Create missing indicators BEFORE imputation ───────────────────────────────
INDICATOR_COLS = [c for c in ['house_size', 'acre_lot', 'bed', 'bath'] if c in df.columns]

for col in INDICATOR_COLS:
    df[f'{col}_was_missing'] = df[col].isnull().astype(int)

print('Missing indicator columns created:')
print(df[[f'{c}_was_missing' for c in INDICATOR_COLS]].sum().to_string())

In [ ]:
# ── Hierarchical Imputation ───────────────────────────────────────────────────
def hierarchical_impute(df, col, group1, group2):
    """Impute `col` using group1 median, falling back to group2 median, then global."""
    if col not in df.columns:
        return df
    global_med = df[col].median()
    g1_med = df.groupby(group1)[col].transform('median')
    g2_med = df.groupby(group2)[col].transform('median')
    df[col] = df[col].fillna(g1_med).fillna(g2_med).fillna(global_med)
    return df

for col in ['bed', 'bath', 'house_size', 'acre_lot']:
    if col in df.columns and 'city' in df.columns and 'state' in df.columns:
        missing_before = df[col].isnull().sum()
        df = hierarchical_impute(df, col, 'city', 'state')
        missing_after = df[col].isnull().sum()
        print(f'{col:15s}: {missing_before:>6,} → {missing_after:>6,} missing')

In [ ]:
# ── Parse prev_sold_date ──────────────────────────────────────────────────────
if 'prev_sold_date' in df.columns:
    df['prev_sold_date'] = pd.to_datetime(df['prev_sold_date'], errors='coerce')
    reference_date = pd.Timestamp('2024-01-01')
    df['years_since_sold'] = ((reference_date - df['prev_sold_date']).dt.days / 365.25).clip(lower=0)
    df['has_been_sold']    = df['prev_sold_date'].notna().astype(int)
    print(f"years_since_sold — mean: {df['years_since_sold'].mean():.1f}, null: {df['years_since_sold'].isna().sum():,}")

## 7️⃣ Feature Engineering

> This is the most important section. We create 15+ domain-informed features that significantly boost model performance.

In [ ]:
# ── Room & Size Ratios ────────────────────────────────────────────────────────
if 'bed' in df.columns and 'bath' in df.columns:
    df['total_rooms']       = df['bed'] + df['bath']
    df['bed_to_bath_ratio'] = (df['bed'] / df['bath'].replace(0, np.nan)).fillna(0).clip(upper=10)

if 'house_size' in df.columns:
    df['log_house_size'] = np.log1p(df['house_size'])
    df['is_large_home']  = (df['house_size'] > df['house_size'].quantile(0.75)).astype(int)

    if 'bed' in df.columns:
        df['sqft_per_bedroom']     = (df['house_size'] / df['bed'].replace(0, np.nan)).fillna(0).clip(upper=5000)
        df['size_bed_interaction'] = df['house_size'] * df['bed']

    if 'bath' in df.columns:
        df['size_bath_interaction'] = df['house_size'] * df['bath']

if 'acre_lot' in df.columns:
    df['log_acre_lot'] = np.log1p(df['acre_lot'])
    df['is_large_lot'] = (df['acre_lot'] > df['acre_lot'].quantile(0.75)).astype(int)

print('✅ Room & size features created')

In [ ]:
# ── Luxury Score (composite 0-5) ──────────────────────────────────────────────
luxury_parts = []
if 'bed'        in df.columns: luxury_parts.append((df['bed']        >= 4).astype(int))
if 'bath'       in df.columns: luxury_parts.append((df['bath']       >= 3).astype(int))
if 'house_size' in df.columns: luxury_parts.append((df['house_size'] > df['house_size'].quantile(0.80)).astype(int))
if 'acre_lot'   in df.columns: luxury_parts.append((df['acre_lot']   > df['acre_lot'].quantile(0.80)).astype(int))
if 'bath' in df.columns and 'bed' in df.columns:
    luxury_parts.append((df['bath'] > df['bed']).astype(int))

if luxury_parts:
    df['luxury_score'] = sum(luxury_parts)
    print(f'luxury_score distribution:\n{df["luxury_score"].value_counts().sort_index()}')

In [ ]:
# ── Geographic / Location Features ───────────────────────────────────────────
if 'lat' in df.columns and 'lng' in df.columns:
    CENTER_LAT, CENTER_LNG = 39.5, -98.35
    df['dist_to_center']    = np.sqrt((df['lat'] - CENTER_LAT)**2 + (df['lng'] - CENTER_LNG)**2)
    df['dist_to_east_coast'] = np.abs(df['lng'] - (-75.0))
    df['dist_to_west_coast'] = np.abs(df['lng'] - (-120.0))
    df['coastal_proximity']  = df[['dist_to_east_coast', 'dist_to_west_coast']].min(axis=1)
    print('✅ Geographic distance features created')

In [ ]:
# ── Population & Density Features ────────────────────────────────────────────
if 'zip_density' in df.columns:
    df['log_zip_density'] = np.log1p(df['zip_density'])
    df['density_bucket']  = pd.qcut(
        df['zip_density'].fillna(df['zip_density'].median()),
        q=5, labels=['rural', 'low', 'medium', 'high', 'urban'], duplicates='drop'
    )
    print('✅ Density features created')

if 'zip_population' in df.columns:
    df['log_zip_population'] = np.log1p(df['zip_population'])
    print('✅ Population features created')

In [ ]:
# ── Time-Based Features ───────────────────────────────────────────────────────
if 'years_since_sold' in df.columns:
    df['recently_sold'] = (df['years_since_sold'] < 2).astype(int)
    df['long_unsold']   = (df['years_since_sold'] > 10).astype(int)
    print('✅ Time features created')

In [ ]:
# ── State-Level Price Statistics ──────────────────────────────────────────────
if 'state' in df.columns:
    state_stats = df.groupby('state')['price'].agg(
        state_median_price='median',
        state_mean_price='mean',
        state_price_std='std'
    ).reset_index()
    df = df.merge(state_stats, on='state', how='left')
    df['price_vs_state_median'] = df['price'] / df['state_median_price'].replace(0, np.nan)
    print('✅ State-level price aggregates created')

In [ ]:
# ── Summary of engineered features ───────────────────────────────────────────
new_feats = [c for c in [
    'total_rooms', 'bed_to_bath_ratio', 'sqft_per_bedroom',
    'size_bed_interaction', 'size_bath_interaction',
    'log_house_size', 'log_acre_lot',
    'is_large_home', 'is_large_lot', 'luxury_score',
    'dist_to_center', 'dist_to_east_coast', 'dist_to_west_coast',
    'coastal_proximity', 'log_zip_density', 'density_bucket',
    'log_zip_population', 'recently_sold', 'long_unsold',
    'state_median_price', 'state_mean_price', 'state_price_std',
    'price_vs_state_median'
] if c in df.columns]

print(f'\n✅ Total engineered features: {len(new_feats)}')
print(df[new_feats].head(3).to_string())

## 8️⃣ Feature Selection & Dimensionality Reduction

> We use XGBoost to rank ALL features by importance and keep the top 35. This improves speed, interpretability, and often performance by removing noise.

### 8A. Prepare Selection Dataset

In [ ]:
# ── Drop leakage columns before selection ─────────────────────────────────────
LEAK_COLS = ['price_per_acre', 'price_per_sqft', 'log_price', 'future_price', 'sold_price']
df.drop(columns=[c for c in LEAK_COLS if c in df.columns], inplace=True)

TARGET       = 'price'
EXCLUDE_COLS = [TARGET, 'prev_sold_date', 'zip_code', 'city', 'street',
                'state_full', 'county_name', 'timezone', 'state']

# ── Encode categoricals for selection stage ───────────────────────────────────
df_sel = df.copy()

def target_encode_cv(df, col, target, n_splits=5, smoothing=10, random_state=42):
    """Cross-validated target encoding with smoothing."""
    encoded     = np.full(len(df), np.nan)
    global_mean = df[target].mean()
    kf          = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for train_idx, val_idx in kf.split(df):
        train = df.iloc[train_idx]
        grp   = train.groupby(col)[target].agg(['mean', 'count'])
        grp['smooth'] = (
            (grp['count'] * grp['mean'] + smoothing * global_mean)
            / (grp['count'] + smoothing)
        )
        encoded[val_idx] = df.iloc[val_idx][col].map(grp['smooth']).fillna(global_mean).values

    return encoded

HIGH_CARD_COLS = [c for c in ['zip_code', 'city', 'street'] if c in df_sel.columns]
for col in HIGH_CARD_COLS:
    df_sel[f'{col}_te'] = target_encode_cv(df_sel, col, TARGET)
    print(f'Target-encoded: {col} → {col}_te')

LOW_CARD_COLS = [c for c in df_sel.select_dtypes(include='object').columns
                 if c not in HIGH_CARD_COLS + EXCLUDE_COLS and df_sel[c].nunique() <= 10]
if LOW_CARD_COLS:
    df_sel = pd.get_dummies(df_sel, columns=LOW_CARD_COLS, drop_first=True)
    print(f'OHE applied to: {LOW_CARD_COLS}')

SEL_EXCLUDE  = set(EXCLUDE_COLS + HIGH_CARD_COLS + ['prev_sold_date'])
ALL_FEATURES = [c for c in df_sel.columns
                if c not in SEL_EXCLUDE
                and df_sel[c].dtype in [np.float64, np.float32, np.int64, np.int32, np.uint8, bool]]

X_sel = df_sel[ALL_FEATURES].copy()
y_sel = df_sel[TARGET].copy()

valid_mask = y_sel.notna()
X_sel, y_sel = X_sel[valid_mask], y_sel[valid_mask]
X_sel = X_sel.fillna(X_sel.median(numeric_only=True))

print(f'\nSelection feature matrix: {X_sel.shape}')

### 8B. XGBoost Feature Importance

In [ ]:
# ── Train a quick XGBoost to rank all features ────────────────────────────────
print('Training XGBoost for feature importance ranking...')

selector_xgb = XGBRegressor(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=RANDOM_STATE, verbosity=0
)

X_s_train, X_s_test, y_s_train, y_s_test = train_test_split(
    X_sel, y_sel, test_size=0.2, random_state=RANDOM_STATE
)
selector_xgb.fit(X_s_train, y_s_train)

importance_df = pd.DataFrame({
    'feature':    X_sel.columns,
    'importance': selector_xgb.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

print(f'Total candidate features : {len(importance_df)}')
print(f'\nTop 10 features:\n{importance_df.head(10).to_string(index=False)}')

In [ ]:
# ── Plot Top 35 Feature Importances ──────────────────────────────────────────
top35 = importance_df.head(TOP_N_FEATURES)

fig, ax = plt.subplots(figsize=(11, 10))
bars = ax.barh(top35['feature'][::-1], top35['importance'][::-1],
               color=PALETTE[0], edgecolor='white', alpha=0.85)
ax.set_xlabel('XGBoost Feature Importance (gain)', fontweight='bold')
ax.set_title(f'🎯 Top {TOP_N_FEATURES} Most Important Features', fontsize=14, fontweight='bold')

for bar, val in zip(bars, top35['importance'][::-1]):
    ax.text(bar.get_width() + 0.0002, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Select top features ───────────────────────────────────────────────────────
SELECTED_FEATURES = top35['feature'].tolist()
print(f'✅ Selected top {len(SELECTED_FEATURES)} features')
print(SELECTED_FEATURES)

### 8C. Optional — PCA Comparison

In [ ]:
# ── PCA Explained Variance Plot ───────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

X_std = StandardScaler().fit_transform(X_sel.fillna(0))
pca   = PCA(n_components=min(50, X_sel.shape[1]), random_state=RANDOM_STATE)
pca.fit(X_std)

cumvar = np.cumsum(pca.explained_variance_ratio_) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1),
            pca.explained_variance_ratio_ * 100, color=PALETTE[2], edgecolor='white')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('PCA — Variance per Component', fontweight='bold')

axes[1].plot(range(1, len(cumvar) + 1), cumvar, marker='o', markersize=4,
             color=PALETTE[0], linewidth=2)
axes[1].axhline(y=90, color='red', linestyle='--', linewidth=1, label='90% threshold')
axes[1].axhline(y=95, color='orange', linestyle='--', linewidth=1, label='95% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance (%)')
axes[1].set_title('PCA — Cumulative Variance', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

n90 = np.argmax(cumvar >= 90) + 1
n95 = np.argmax(cumvar >= 95) + 1
print(f'Components for 90% variance: {n90}')
print(f'Components for 95% variance: {n95}')
print(f'\n💡 Decision: Using XGBoost top {TOP_N_FEATURES} features (better interpretability than PCA)')

plt.suptitle('📉 PCA vs Feature Selection Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9️⃣ Data Splitting, Encoding & Scaling

> We build the final feature matrix using only the selected top 35 features, then apply Target Encoding + RobustScaler.

In [ ]:
# ── Build final df with selected features only ────────────────────────────────
MISS_IND_COLS   = [c for c in df_sel.columns if c.endswith('_was_missing')]
FINAL_FEATURES  = [c for c in SELECTED_FEATURES + MISS_IND_COLS if c in df_sel.columns]
FINAL_FEATURES  = list(dict.fromkeys(FINAL_FEATURES))  # deduplicate, preserve order

print(f'Final feature count: {len(FINAL_FEATURES)}')
print(f'  → top selected   : {len(SELECTED_FEATURES)}')
print(f'  → miss indicators: {len(MISS_IND_COLS)}')

In [ ]:
# ── Build X, y ────────────────────────────────────────────────────────────────
X = df_sel[FINAL_FEATURES].copy()
y = df_sel[TARGET].copy()

valid_mask = y.notna()
X, y = X[valid_mask], y[valid_mask]
X = X.fillna(X.median(numeric_only=True))

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'y range: ${y.min():,.0f} – ${y.max():,.0f}')

In [ ]:
# ── Train/Test Split (80/20) ──────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

In [ ]:
# ── RobustScaler (robust to outliers) ────────────────────────────────────────
scaler         = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=FINAL_FEATURES, index=X_train.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=FINAL_FEATURES, index=X_test.index)

print('✅ RobustScaler fitted on training data')

In [ ]:
# ── GroupKFold setup (grouped by ZIP for realistic geo-validation) ────────────
groups_train = df_sel.loc[X_train.index, 'zip_code'].values
gkf          = GroupKFold(n_splits=N_SPLITS)
print(f'✅ GroupKFold ({N_SPLITS} folds) ready — grouped by zip_code')

## 🔟 Model Training

> We train XGBoost and LightGBM with GroupKFold cross-validation. A baseline (median predictor) is included for reference.

In [ ]:
# ── Evaluation helper ─────────────────────────────────────────────────────────
def evaluate_model(model, X, y, groups, cv, label='Model', verbose=True):
    """GroupKFold CV → returns dict of mean metrics."""
    r2_l, mae_l, rmse_l, mape_l = [], [], [], []

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y, groups)):
        Xtr, Xval = X.iloc[tr_idx], X.iloc[val_idx]
        ytr, yval = y.iloc[tr_idx], y.iloc[val_idx]

        model.fit(Xtr, ytr)
        preds = model.predict(Xval)

        r2_l.append(r2_score(yval, preds))
        mae_l.append(mean_absolute_error(yval, preds))
        rmse_l.append(np.sqrt(mean_squared_error(yval, preds)))
        mape_l.append(np.mean(np.abs((yval - preds) / yval.clip(lower=1))) * 100)

    result = {
        'Model':  label,
        'R2':     np.mean(r2_l),
        'R2_std': np.std(r2_l),
        'MAE':    np.mean(mae_l),
        'RMSE':   np.mean(rmse_l),
        'MAPE_%': np.mean(mape_l),
    }
    if verbose:
        print(f"  {label:30s}  R²={result['R2']:.4f}±{result['R2_std']:.4f}  "
              f"MAE=${result['MAE']:>10,.0f}  MAPE={result['MAPE_%']:.1f}%")
    return result, model

In [ ]:
baseline  = DummyRegressor(strategy='median')
res_base, _ = evaluate_model(baseline, X_train_scaled, y_train, groups_train, gkf, 'Baseline (Median)')
all_results = [res_base]

In [ ]:
# ── XGBoost ───────────────────────────────────────────────────────────────────
xgb_model = XGBRegressor(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
    n_jobs=-1, random_state=RANDOM_STATE, verbosity=0
)
res_xgb, xgb_fitted = evaluate_model(xgb_model, X_train_scaled, y_train,
                                      groups_train, gkf, 'XGBoost')
all_results.append(res_xgb)

In [ ]:
# ── LightGBM ──────────────────────────────────────────────────────────────────
lgbm_model = LGBMRegressor(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    num_leaves=63, min_child_samples=10,
    reg_alpha=0.1, reg_lambda=1.0,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
)
res_lgbm, lgbm_fitted = evaluate_model(lgbm_model, X_train_scaled, y_train,
                                        groups_train, gkf, 'LightGBM')
all_results.append(res_lgbm)

## 1️⃣1️⃣ Hyperparameter Tuning (Optuna)

> We run 50 Optuna trials to find the best LightGBM hyperparameters. This takes ~5 minutes.

In [ ]:
# ── Optuna objective ──────────────────────────────────────────────────────────
def lgbm_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 800),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 200),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 80),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'n_jobs': -1, 'random_state': RANDOM_STATE, 'verbose': -1,
    }
    model = LGBMRegressor(**params)
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    return np.sqrt(mean_squared_error(y_test, preds))


study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study.optimize(lgbm_objective, n_trials=50, show_progress_bar=True)

print(f'\n🏆 Best RMSE : ${study.best_value:,.0f}')
print('Best params :', study.best_params)

In [ ]:
# ── Optuna optimization history ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

trial_rmse  = [t.value for t in study.trials if t.value is not None]
best_so_far = np.minimum.accumulate(trial_rmse)

axes[0].plot(trial_rmse, alpha=0.5, color=PALETTE[1], label='Trial RMSE')
axes[0].plot(best_so_far, color=PALETTE[0], linewidth=2, label='Best so far')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('RMSE ($)')
axes[0].set_title('Optuna Optimization History', fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[0].legend()

param_importance = optuna.importance.get_param_importances(study)
axes[1].barh(list(param_importance.keys())[::-1],
             list(param_importance.values())[::-1],
             color=PALETTE[2], edgecolor='white')
axes[1].set_title('Optuna Hyperparameter Importance', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.suptitle('🔬 Optuna Tuning Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Train tuned LightGBM with best params ─────────────────────────────────────
best_params = dict(study.best_params)
best_params.update({'n_jobs': -1, 'random_state': RANDOM_STATE, 'verbose': -1})

tuned_lgbm = LGBMRegressor(**best_params)
res_tuned, tuned_fitted = evaluate_model(tuned_lgbm, X_train_scaled, y_train,
                                         groups_train, gkf, 'LightGBM (Tuned)')
all_results.append(res_tuned)

## 1️⃣2️⃣ Model Comparison & Final Model

> Full side-by-side comparison of all models. We select the best and refit on the complete training set.

In [ ]:
# ── Comparison Table ──────────────────────────────────────────────────────────
results_df = (
    pd.DataFrame(all_results)
    .set_index('Model')
    .sort_values('R2', ascending=False)
    .round(4)
)

print('\n' + '='*65)
print('  📊  MODEL PERFORMANCE COMPARISON (GroupKFold CV)')
print('='*65)
print(results_df[['R2', 'R2_std', 'MAE', 'RMSE', 'MAPE_%']].to_string())
print('='*65)

results_df.style \
    .background_gradient(subset=['R2'], cmap='Greens') \
    .background_gradient(subset=['MAE', 'RMSE'], cmap='Reds_r') \
    .format({'R2': '{:.4f}', 'R2_std': '±{:.4f}',
             'MAE': '${:,.0f}', 'RMSE': '${:,.0f}', 'MAPE_%': '{:.2f}%'})

In [ ]:
# ── Comparison Bar Charts ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = [('R2', 'R² Score (↑)', PALETTE[0], True),
           ('MAE', 'MAE in $ (↓)', PALETTE[1], False),
           ('RMSE', 'RMSE in $ (↓)', PALETTE[2], False)]

for ax, (metric, title, color, higher) in zip(axes, metrics):
    vals = results_df[metric].sort_values(ascending=not higher)
    bars = ax.barh(vals.index, vals.values, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontweight='bold', fontsize=12)
    if metric in ('MAE', 'RMSE'):
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    for bar in bars:
        w = bar.get_width()
        ax.text(w + w * 0.01, bar.get_y() + bar.get_height() / 2,
                f'${w/1e3:.0f}K' if metric in ('MAE','RMSE') else f'{w:.3f}',
                va='center', fontsize=8)

plt.suptitle('🏆 Model Comparison Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Select and refit best model on full training data ─────────────────────────
best_model_name = results_df['R2'].idxmax()
print(f'🏆 Best model selected: {best_model_name}')

if best_model_name == 'LightGBM (Tuned)':
    final_model = LGBMRegressor(**best_params)
elif best_model_name == 'XGBoost':
    final_model = XGBRegressor(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        n_jobs=-1, random_state=RANDOM_STATE, verbosity=0
    )
else:
    final_model = LGBMRegressor(**best_params)

final_model.fit(X_train_scaled, y_train)
print('✅ Final model fitted on full training set')

In [ ]:
# ── Hold-out Test Set Evaluation ──────────────────────────────────────────────
y_pred_test = final_model.predict(X_test_scaled)
residuals   = y_test - y_pred_test

r2_test   = r2_score(y_test, y_pred_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
mape_test = np.mean(np.abs((y_test - y_pred_test) / y_test.clip(lower=1))) * 100

print(f'\n📐 TEST SET METRICS ({best_model_name})')
print(f'  R²   : {r2_test:.4f}')
print(f'  MAE  : ${mae_test:,.0f}')
print(f'  RMSE : ${rmse_test:,.0f}')
print(f'  MAPE : {mape_test:.2f}%')

In [ ]:
# ── Evaluation Plots ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_test, y_pred_test, alpha=0.25, s=6, color=PALETTE[0])
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5)
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[0].set_title(f'Actual vs Predicted\nR²={r2_test:.4f}', fontweight='bold')

axes[1].scatter(y_pred_test, residuals, alpha=0.25, s=6, color=PALETTE[2])
axes[1].axhline(0, color='red', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Predicted Price')
axes[1].set_ylabel('Residual')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[1].set_title('Residuals vs Predicted', fontweight='bold')

axes[2].hist(residuals, bins=70, color=PALETTE[1], edgecolor='white', alpha=0.85)
axes[2].axvline(0, color='red', linewidth=1.5)
axes[2].set_xlabel('Residual ($)')
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[2].set_title('Residual Distribution', fontweight='bold')

plt.suptitle(f'📊 {best_model_name} — Test Set Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP Feature Importance ───────────────────────────────────────────────────
print('Computing SHAP values (may take ~1 min)...')
shap_sample = X_test_scaled.sample(min(2000, len(X_test_scaled)), random_state=RANDOM_STATE)
explainer   = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(shap_sample)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, shap_sample, plot_type='bar', max_display=20, show=False)
plt.title('SHAP Feature Importance (Mean |SHAP value|)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 9))
shap.summary_plot(shap_values, shap_sample, max_display=20, show=False)
plt.title('SHAP Summary Beeswarm — Feature Impact on Price', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

## 1️⃣3️⃣ Save Artifacts

> All artifacts are saved to the `artifacts/` folder for deployment.

In [ ]:
# ── Refit final model on ALL available data (train + test) ───────────────────
X_full = pd.concat([X_train_scaled, X_test_scaled])
y_full = pd.concat([y_train, y_test])
final_model.fit(X_full, y_full)
print('✅ Final model refitted on full dataset (train + test)')

In [ ]:
# ── Save all artifacts ────────────────────────────────────────────────────────
joblib.dump(final_model,       'artifacts/trained_model.joblib')
joblib.dump(scaler,            'artifacts/preprocessor.joblib')
# NOTE: app.py uses 'scaler.joblib' — save under both names for compatibility
joblib.dump(scaler,            'artifacts/scaler.joblib')
joblib.dump(SELECTED_FEATURES, 'artifacts/selected_features.joblib')

feature_info = {
    'selected_features': SELECTED_FEATURES,
    'final_features':    FINAL_FEATURES,
    'feature_cols':      FINAL_FEATURES,   # alias used by app.py
    'miss_ind_cols':     MISS_IND_COLS,
    'high_card_cols':    HIGH_CARD_COLS,
    'low_card_cols':     LOW_CARD_COLS if 'LOW_CARD_COLS' in dir() else [],
    'target':            TARGET,
    'n_selected':        len(SELECTED_FEATURES),
    'n_final':           len(FINAL_FEATURES),
    'best_model':        best_model_name,
    'test_r2':           round(r2_test, 4),
    'test_mae':          round(mae_test, 2),
    'test_rmse':         round(rmse_test, 2),
    'test_mape_pct':     round(mape_test, 2),
}
joblib.dump(feature_info, 'artifacts/feature_info.joblib')
with open('artifacts/feature_info.json', 'w') as f:
    json.dump(feature_info, f, indent=2)

results_df.reset_index().to_csv('artifacts/metrics.csv', index=False)
zip_df.to_csv('artifacts/zip_lookup.csv', index=False)
importance_df.to_csv('artifacts/feature_importance.csv', index=False)

print('\n📦 Artifacts saved:')
for f in sorted(os.listdir('artifacts')):
    path = os.path.join('artifacts', f)
    size = os.path.getsize(path)
    print(f'   {f:40s}  {size/1024:>8.1f} KB')

## 1️⃣4️⃣ Prediction Function

> A reusable function that takes raw property data as a dict and returns a predicted price.

In [ ]:
def predict_price(input_data: dict,
                  model_path:        str = 'artifacts/trained_model.joblib',
                  scaler_path:       str = 'artifacts/preprocessor.joblib',
                  feature_info_path: str = 'artifacts/feature_info.joblib',
                  zip_lookup_path:   str = 'artifacts/zip_lookup.csv') -> dict:
    """
    Predict house price from raw input dictionary.

    Parameters
    ----------
    input_data : dict
        Keys: bed, bath, house_size, acre_lot, zip_code (+ any other features)

    Returns
    -------
    dict with keys: predicted_price, predicted_price_formatted, confidence_range
    """
    model        = joblib.load(model_path)
    scaler       = joblib.load(scaler_path)
    feature_info = joblib.load(feature_info_path)
    zip_lookup   = pd.read_csv(zip_lookup_path, dtype={'zip_code': str})

    FINAL_FEATURES = feature_info['final_features']
    row = pd.DataFrame([input_data])

    if 'zip_code' in row.columns:
        row['zip_code'] = row['zip_code'].astype(str).str.zfill(5)
        row = row.merge(zip_lookup, on='zip_code', how='left')

    if 'bed' in row.columns and 'bath' in row.columns:
        row['total_rooms']       = row['bed'] + row['bath']
        row['bed_to_bath_ratio'] = (row['bed'] / row['bath'].replace(0, np.nan)).fillna(0)

    if 'house_size' in row.columns:
        row['log_house_size'] = np.log1p(row['house_size'])
        if 'bed' in row.columns:
            row['sqft_per_bedroom']    = (row['house_size'] / row['bed'].replace(0, np.nan)).fillna(0)
            row['size_bed_interaction'] = row['house_size'] * row['bed']
        if 'bath' in row.columns:
            row['size_bath_interaction'] = row['house_size'] * row['bath']

    if 'acre_lot' in row.columns:
        row['log_acre_lot'] = np.log1p(row['acre_lot'])

    if 'lat' in row.columns and 'lng' in row.columns:
        row['dist_to_center']    = np.sqrt((row['lat'] - 39.5)**2 + (row['lng'] - (-98.35))**2)
        row['coastal_proximity'] = min(abs(row['lng'].values[0] - (-75.0)),
                                       abs(row['lng'].values[0] - (-120.0)))

    if 'zip_density'    in row.columns: row['log_zip_density']    = np.log1p(row['zip_density'])
    if 'zip_population' in row.columns: row['log_zip_population'] = np.log1p(row['zip_population'])

    lux = 0
    if 'bed'        in row.columns and row['bed'].values[0]        >= 4:  lux += 1
    if 'bath'       in row.columns and row['bath'].values[0]       >= 3:  lux += 1
    if 'house_size' in row.columns and row['house_size'].values[0] > 3000: lux += 1
    if 'acre_lot'   in row.columns and row['acre_lot'].values[0]   > 1.0:  lux += 1
    row['luxury_score'] = lux

    for col in feature_info.get('miss_ind_cols', []):
        if col not in row.columns:
            row[col] = 0

    for col in FINAL_FEATURES:
        if col not in row.columns:
            row[col] = 0
    row = row[FINAL_FEATURES]

    row_scaled = scaler.transform(row)
    price      = float(model.predict(row_scaled)[0])
    lo, hi     = price * 0.90, price * 1.10

    return {
        'predicted_price':           round(price, 2),
        'predicted_price_formatted': f'${price:,.0f}',
        'confidence_range':          f'${lo:,.0f} – ${hi:,.0f}',
    }


# ── Demo prediction ───────────────────────────────────────────────────────────
sample_house = {'bed': 4, 'bath': 3, 'house_size': 2800, 'acre_lot': 0.25, 'zip_code': '10001'}
result = predict_price(sample_house)
print('🏠 Sample Prediction')
print(f'   Input    : {sample_house}')
print(f'   Predicted: {result["predicted_price_formatted"]}')
print(f'   Range    : {result["confidence_range"]}')

## ✅ Final Summary

In [ ]:
best_row = results_df.loc[best_model_name]

print()
print('╔' + '═'*60 + '╗')
print('║  🏠  PropIQ — USA House Price Prediction  SUMMARY        ║')
print('╠' + '═'*60 + '╣')
print(f'║  Dataset rows          : {len(df):>12,}                  ║')
print(f'║  Selected features     : {len(SELECTED_FEATURES):>12}  (from {len(importance_df)} candidates)    ║')
print(f'║  Final feature count   : {len(FINAL_FEATURES):>12}                  ║')
print(f'║  ZIP code enrichment   : {len(zip_df):>12,} records             ║')
print(f'║  Validation            :   GroupKFold (zip_code, {N_SPLITS} folds)  ║')
print('╠' + '═'*60 + '╣')
print(f'║  Best model            : {best_model_name:<35}║')
print(f'║  CV R²                 : {best_row["R2"]:>12.4f}                  ║')
print(f'║  CV MAE                : ${best_row["MAE"]:>11,.0f}                  ║')
print(f'║  CV RMSE               : ${best_row["RMSE"]:>11,.0f}                  ║')
print(f'║  CV MAPE               : {best_row["MAPE_%"]:>11.2f}%                  ║')
print('╠' + '═'*60 + '╣')
print('║  Artifacts saved:                                        ║')
print('║    artifacts/trained_model.joblib                        ║')
print('║    artifacts/preprocessor.joblib                         ║')
print('║    artifacts/scaler.joblib  (app.py alias)               ║')
print('║    artifacts/selected_features.joblib                    ║')
print('║    artifacts/feature_info.joblib + .json                 ║')
print('║    artifacts/metrics.csv                                 ║')
print('║    artifacts/zip_lookup.csv                              ║')
print('║    artifacts/feature_importance.csv                      ║')
print('╚' + '═'*60 + '╝')
print()
print('🎉 Artifacts Saved Successfully — PropIQ Pipeline Complete!')
print('\nTo launch the Streamlit app:')
print('   streamlit run app.py')